# DAT09 · 미션7 검토한 후보를 승인하고 이력 남기기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 3시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09.md`를 연결하세요.
- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션6을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m06.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m07_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=6):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 6)
else:
    print('DAT09.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션7 검토한 후보를 승인하고 이력 남기기

예상 3시간입니다.

#### 이번 미션의 목표

후보 저장과 사람의 승인을 분리하고, 권한·근거·버전을 확인한 값만 확정하며 승인 이력을 검증합니다.

### 1단계 승인 상태와 담당자 권한 구현하기

#### 이 단계에서 하는 일

후보 수량이 그럴듯해도 제안한 프로그램이 스스로 승인하면 검토 절차가 사라집니다. 누가 어떤 상태를 바꿀 수 있는지 정하고 DB 변경 직전에 검사해야 합니다.

#### 시작 전 확인

미션6까지 완료한 결과를 사용합니다. 노트북의 m07_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

config/approval_policy.json, actor_registry.json과 미션3 DB를 사용합니다. analyst는 분석요원, staff는 군수참모, steward는 데이터관리관, service는 자동 제안 프로그램의 모의 역할입니다.

#### 작업 순서

- 제안은 분석요원과 서비스, 고영향 확인은 군수참모, 승인·반려는 관리관만 할 수 있게 ApprovalWorkflow를 작성하세요. 요청에 적힌 역할 대신 등록된 actor_id의 역할을 확인합니다.

- 일반 후보는 PROPOSED에서 APPROVED 또는 REJECTED로 바뀝니다. 원출처 수량의 최댓값-최솟값이 100EA 이상인 후보는 참모가 REVIEWED로 확인한 뒤 승인하게 합니다. 정확히 100EA인 경우도 포함합니다.

- 제안자는 자기 후보를 승인하지 못하게 하세요. 승인 직전 후보 버전과 원관측 버전, 근거·사유, 미종결 상태를 검사합니다.

- 후보 상태 변경·승인 마스터 기록·감사 기록은 한 트랜잭션에서 함께 성공하거나 함께 취소되게 하세요. 외부 체계 반영은 연결하지 않습니다.


#### 저장할 결과

state_machine.md에 상태·역할·조건을 적고 ApprovalWorkflow를 c에 저장합니다. DB에는 후보·승인 마스터·감사 기록을 분리합니다.

#### 직접 확인하는 방법

범위 99EA는 일반, 100EA는 참모 확인이 필요한 후보입니다. 참모 확인만으로 승인 마스터에 들어가면 안 됩니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M07/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M07_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 후보의 원자료와 버전을 함께 저장하기

#### 이 단계에서 하는 일

나중에 원보고가 바뀌었는데 예전 후보를 승인하면 잘못된 값이 확정될 수 있습니다. 후보가 어떤 자료의 어느 버전을 근거로 만들어졌는지 함께 저장합니다. 후보가 없는 대상도 기록에서 빠뜨리지 않습니다.

#### 시작 전 확인

미션7의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m07_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션4의 c['candidates']와 c['groups'], 앞 단계의 approval을 사용합니다. 기본 자료에는 값 충돌 또는 비교 불가인 대상 65개가 있습니다.

#### 작업 순서

- 65개 대상마다 후보값 유무를 확인하세요. 후보값이 있으면 SERVICE_1의 제안으로 저장하고 PROPOSED로 남기며, 없으면 NO_CANDIDATE로 대장에 남깁니다.

- 후보 ID·거점·품목·기준시각·후보 수량·출처별 관측 버전·후보 버전·제안자·근거를 저장하세요. 원보고 ID는 미션4 후보 근거와 연결해 확인할 수 있게 합니다.

- 제안 근거가 없거나 수량이 유효하지 않으면 제안 성공으로 처리하지 마세요. 원자료 값과 후보값을 다른 항목에 유지합니다.

- 이 단계에서는 승인 함수를 호출하지 않습니다. 승인 마스터 건수와 외부 반영 상태를 조회해 자동 확정이 없는지 확인하세요.


#### 저장할 결과

proposal_ledger.jsonl과 DB 후보 표를 저장합니다. 대장의 대상 수와 DB에 실제 저장된 제안 수는 따로 확인합니다.

#### 직접 확인하는 방법

기본 대장의 대상은 65개입니다. 그중 후보 없는 대상이 있으면 DB 제안 수가 대장 수보다 적을 수 있습니다. 기본 승인 마스터는 0건이어야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M07/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M07_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 승인과 반려 및 대기의 네 경로 실행하기

#### 이 단계에서 하는 일

문서에 승인 절차를 적는 것과 실제 상태가 바뀌는 것은 다릅니다. 정상 경로를 직접 실행하되 기본 자료의 후보를 승인해 버리지 않도록 시험 DB를 분리합니다.

#### 시작 전 확인

미션7의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m07_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/approval_cases.json의 네 사례와 ApprovalWorkflow를 사용합니다. 임시 SQLite DB를 만들고 시험이 끝나면 기본 DB의 승인 건수가 그대로인지 확인합니다.

#### 작업 순서

- 일반 후보를 제안하고 관리관이 사유를 적어 승인하는 경로를 실행하세요. 후보 상태와 승인 마스터 기록을 조회합니다.

- 고영향 후보는 참모 확인 후 관리관 승인을 실행하세요. 참모 확인 기록과 최종 승인 기록이 모두 남아야 합니다.

- 관리관 반려 사례는 REJECTED로, 근거 확인을 기다리는 사례는 PROPOSED로 유지하세요. 둘은 승인 마스터에 들어가면 안 됩니다.

- 사례별 예상 상태·실제 상태·판정·외부 반영 상태를 기록하세요. 승인할 때 expected_version으로 검토한 후보 버전을 넘기고 등록된 원관측 버전도 확인합니다.


#### 저장할 결과

decision_ledger.jsonl에 네 경로의 결과를 저장하고, 기본 DB와 분리된 시험 코드·DB 경로를 남깁니다.

#### 직접 확인하는 방법

일반 승인·참모 확인 후 승인·반려·대기의 네 상태가 사례 파일의 예상과 같아야 합니다. 시험 후 기본 승인 마스터는 여전히 0건입니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M07/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M07_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 승인 조건을 어긴 열 가지 요청 차단하기

#### 이 단계에서 하는 일

정상 승인만 시험하면 권한을 우회한 요청이나 오래된 후보 승인 오류를 놓칠 수 있습니다. 잘못된 요청을 각각 독립된 DB에서 실행하고, 마스터가 바뀌지 않았다는 사실과 거절 이유를 함께 확인합니다.

#### 시작 전 확인

미션7의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m07_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/approval_attempts.json의 열 사례와 ApprovalWorkflow를 사용합니다. 앞 시험의 승인 결과가 뒤 시험을 오염시키지 않도록 매번 새 후보와 DB를 만듭니다.

#### 작업 순서

- 무권한 요청·본인 승인·빈 근거·고영향 참모 확인 누락·옛 후보 버전의 다섯 요청을 각각 실행하세요.

- 원관측 버전 변경·중복 승인·반려 후 재승인·요청 역할 위조·서비스 자동 승인도 각각 실행하세요. 중복 승인은 먼저 정상 승인한 뒤 같은 승인을 다시 요청합니다.

- 각 거절 요청 직전과 직후의 승인 마스터 건수를 비교하세요. 중복 승인 사례처럼 시험 준비로 이미 있는 승인값은 유지하되 추가 변경은 없어야 합니다.

- 실제 거절 반환값과 감사 기록의 DENIED·거절 이유를 확인해 저장하세요. 함수가 단지 False를 반환했다는 사실만으로 끝내지 않습니다.


#### 저장할 결과

approval_attack_results.csv에 사례명·거절 여부·마스터 불변·거절 기록 여부를 저장합니다.

#### 직접 확인하는 방법

열 요청 모두 거절되고 마스터 추가 변경이 0건이어야 합니다. 거절 시도 10건 모두 감사 기록이 남아야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M07/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M07_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 감사 기록의 수정과 삭제 탐지하기

#### 이 단계에서 하는 일

기록 파일이 있다는 사실만으로 이력이 보존됐다고 볼 수 없습니다. 내용 변경과 중간 삭제뿐 아니라 끝부분 삭제도 찾을 수 있는지 시험합니다. 해시 검증은 교육용 탐지 장치이며 실제 인증·접근통제를 대신하지 않습니다.

#### 시작 전 확인

미션7의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m07_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

approval의 감사 기록과 SHA256 함수를 사용합니다. 원본은 보존하고 네 가지 시험용 목록을 별도로 만듭니다.

#### 작업 순서

- 각 기록에 이전 기록의 해시와 현재 내용을 함께 넣어 새 해시를 계산하세요. 기록에는 순번·후보 ID·행위자·역할·행동·결과·사유·시각을 남깁니다.

- 정상 이력의 전체 건수와 마지막 해시를 audit_anchor.json에 따로 저장하세요. 기준점은 검증 대상 이력과 함께 임의 수정하면 안 됩니다.

- 정상 이력, 중간 기록의 사유 변경, 중간 행 삭제, 마지막 행 삭제의 네 시험을 실행하세요. 앞에서부터 해시를 다시 계산하고 마지막 건수·해시도 기준점과 맞춥니다.

- 원본은 통과하고 세 변조본은 실패하는지 기록하세요. 공격자가 이력과 기준점을 모두 다시 쓸 수 있다면 이 방법만으로 막을 수 없다는 한계도 적습니다.


#### 저장할 결과

audit.jsonl, audit_anchor.json, tamper_test.json과 검증 함수를 저장합니다.

#### 직접 확인하는 방법

정상 1종은 검증 성공, 수정·중간 삭제·끝 삭제의 3종은 탐지돼야 합니다. 끝 삭제는 기준점의 건수·마지막 해시 비교가 필요합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M07/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M07_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 승인 상태와 담당자 권한 구현하기
def m07_s1(c):
    """승인 상태와 담당자 권한 구현하기."""
    import sqlite3
    out = folder(c, 7)

    class ApprovalWorkflow:
        def __init__(self, db_path, policy_path, actor_path):
            self.db_path = Path(db_path)
            self.policy = read_json(policy_path)
            self.actors = read_json(actor_path)
            self.audit_seq = 0
            self.audit_records = []
            self.last_hash = '0' * 64

        def _conn(self):
            conn = sqlite3.connect(self.db_path)
            conn.row_factory = sqlite3.Row
            return conn

        def _role(self, actor_id):
            return self.actors.get(actor_id)

        def _record_audit(self, cand_id, actor_id, action, result, reason, ts='2026-10-01T15:10:00+00:00'):
            self.audit_seq += 1
            payload = {
                'seq': self.audit_seq, 'candidate_id': cand_id, 'actor_id': actor_id,
                'role': self._role(actor_id) or 'unknown', 'action': action,
                'result': result, 'reason': reason, 'timestamp': ts,
                'prev_hash': self.last_hash,
            }
            cur_hash = sha(payload)
            payload['hash'] = cur_hash
            self.last_hash = cur_hash
            self.audit_records.append(payload)
            return payload

        def propose(self, cand_id, site_id, item_id, metric, as_of, candidate_ea, actor_id, reason, source_versions, candidate_version=1, source_quantities=None):
            role = self._role(actor_id)
            if role not in ('analyst', 'service'):
                self._record_audit(cand_id, actor_id, 'propose', 'DENIED', '제안 권한 없음')
                return False, 'DENIED: propose role required'
            if not reason or candidate_ea is None:
                self._record_audit(cand_id, actor_id, 'propose', 'DENIED', '유효 수량 또는 근거 누락')
                return False, 'DENIED: missing quantity or reason'
            sq = [q for q in (source_quantities or [candidate_ea]) if q is not None]
            high_impact = (max(sq) - min(sq) >= self.policy['high_impact_range_ea']) if len(sq) >= 2 else False
            with self._conn() as conn:
                conn.execute(
                    """INSERT OR REPLACE INTO candidate(
                        candidate_id, site_id, item_id, metric, as_of, candidate_ea, status,
                        proposer_id, reason, source_versions, candidate_version, reviewed_by
                    ) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
                    (cand_id, site_id, item_id, metric, as_of, int(candidate_ea), 'PROPOSED',
                     actor_id, reason, json.dumps(source_versions, ensure_ascii=False, sort_keys=True),
                     int(candidate_version), 'HIGH_IMPACT_REQUIRED' if high_impact else ''),
                )
            self._record_audit(cand_id, actor_id, 'propose', 'ALLOWED', reason)
            return True, 'PROPOSED'

        def review_high_impact(self, cand_id, actor_id, reason):
            if self._role(actor_id) != 'staff':
                self._record_audit(cand_id, actor_id, 'review_high_impact', 'DENIED', '군수참모(staff) 권한 필요')
                return False, 'DENIED: staff required'
            if not reason or not str(reason).strip():
                self._record_audit(cand_id, actor_id, 'review_high_impact', 'DENIED', '근거 누락')
                return False, 'DENIED: reason required'
            with self._conn() as conn:
                row = conn.execute('SELECT * FROM candidate WHERE candidate_id=?', (cand_id,)).fetchone()
                if not row or row['status'] != 'PROPOSED':
                    self._record_audit(cand_id, actor_id, 'review_high_impact', 'DENIED', 'PROPOSED 상태 아님')
                    return False, 'DENIED: invalid state'
                conn.execute('UPDATE candidate SET status=?, reviewed_by=? WHERE candidate_id=?', ('REVIEWED', actor_id, cand_id))
            self._record_audit(cand_id, actor_id, 'review_high_impact', 'ALLOWED', reason)
            return True, 'REVIEWED'

        def decide(self, cand_id, actor_id, action, reason, expected_version=1, current_source_versions=None, claimed_role=None):
            role = self._role(actor_id)
            if claimed_role and claimed_role != role:
                self._record_audit(cand_id, actor_id, action, 'DENIED', f'요청 역할 위조({claimed_role} != {role})')
                return False, 'DENIED: role mismatch'
            if role != 'steward' or action not in ('approve', 'reject'):
                self._record_audit(cand_id, actor_id, action, 'DENIED', '데이터관리관(steward) 승인/반려 권한 필요')
                return False, 'DENIED: steward required'
            if not reason or not str(reason).strip():
                self._record_audit(cand_id, actor_id, action, 'DENIED', '근거 사유 누락')
                return False, 'DENIED: reason required'
            with self._conn() as conn:
                row = conn.execute('SELECT * FROM candidate WHERE candidate_id=?', (cand_id,)).fetchone()
                if not row:
                    self._record_audit(cand_id, actor_id, action, 'DENIED', '후보 없음')
                    return False, 'DENIED: missing candidate'
                if row['proposer_id'] == actor_id:
                    self._record_audit(cand_id, actor_id, action, 'DENIED', '제안자 본인 승인 금지')
                    return False, 'DENIED: self-approval forbidden'
                if row['status'] in ('APPROVED', 'REJECTED'):
                    self._record_audit(cand_id, actor_id, action, 'DENIED', f"이미 종결된 상태({row['status']})")
                    return False, 'DENIED: already decided'
                if int(row['candidate_version']) != int(expected_version):
                    self._record_audit(cand_id, actor_id, action, 'DENIED', f"후보 버전 불일치({row['candidate_version']}!={expected_version})")
                    return False, 'DENIED: stale candidate version'
                if current_source_versions is not None and json.loads(row['source_versions']) != current_source_versions:
                    self._record_audit(cand_id, actor_id, action, 'DENIED', '원관측 버전 변경됨')
                    return False, 'DENIED: observation version changed'
                if action == 'approve' and row['reviewed_by'] == 'HIGH_IMPACT_REQUIRED' and row['status'] != 'REVIEWED':
                    self._record_audit(cand_id, actor_id, action, 'DENIED', '고영향 후보 참모(staff) 확인 누락')
                    return False, 'DENIED: staff review required'
                new_state = 'APPROVED' if action == 'approve' else 'REJECTED'
                conn.execute('UPDATE candidate SET status=? WHERE candidate_id=?', (new_state, cand_id))
                if new_state == 'APPROVED':
                    conn.execute(
                        """INSERT INTO approved_master(site_id, item_id, metric, as_of, approved_ea, candidate_id, steward_id, reason, approved_at, external_write)
                        VALUES (?,?,?,?,?,?,?,?,?,?)""",
                        (row['site_id'], row['item_id'], row['metric'], row['as_of'], row['candidate_ea'],
                         cand_id, actor_id, reason, '2026-10-01T15:10:00+00:00', self.policy['external_write']),
                    )
            self._record_audit(cand_id, actor_id, action, 'ALLOWED', reason)
            return True, new_state

        def master_count(self):
            with self._conn() as conn:
                return conn.execute('SELECT COUNT(*) FROM approved_master').fetchone()[0]

    (out / 'state_machine.md').write_text(
        '# M07 승인 상태기계\n'
        '- 일반 후보: `PROPOSED` -> (steward) -> `APPROVED` 또는 `REJECTED`\n'
        '- 고영향 후보(범위 `>=100EA`): `PROPOSED` -> (staff) -> `REVIEWED` -> (steward) -> `APPROVED`/`REJECTED`\n'
        '- 본인 승인 금지, 근거 필수, 후보·원관측 버전 검증, 외부 반영 `NOT_CONNECTED`.\n',
        encoding='utf-8',
    )
    approval = ApprovalWorkflow(c['db'], c['root'] / 'config/approval_policy.json', c['root'] / 'config/actor_registry.json')
    c['ApprovalWorkflow'] = ApprovalWorkflow
    c['approval'] = approval
    return [
        {'state': 'PROPOSED', 'actor_roles': 'analyst, service', 'condition': '근거·원관측 버전 포함 제안'},
        {'state': 'REVIEWED', 'actor_roles': 'staff', 'condition': '수량 차이 범위 >= 100EA 고영향 확인'},
        {'state': 'APPROVED', 'actor_roles': 'steward', 'condition': '타인 제안·버전 일치·사유 입력 시 확정'},
        {'state': 'REJECTED', 'actor_roles': 'steward', 'condition': '사유 입력 후 반려(마스터 미반영)'},
    ]

# STEP 02 후보의 원자료와 버전을 함께 저장하기
def m07_s2(c):
    """후보의 원자료와 버전을 함께 저장하기."""
    out = folder(c, 7)
    wf = c['approval']
    ledger = []
    for cand in c['candidates']:
        sv = json.loads(cand['source_versions'])
        if cand['status'] == 'PROPOSED':
            grp = c['groups'][(cand['site_id'], cand['item_id'], cand['metric'], cand['as_of'])]
            sqs = [r['quantity_ea'] for r in grp['rows'] if r['status'] == 'ACCEPTED' and r['quantity_ea'] is not None]
            ok, msg = wf.propose(
                cand['candidate_id'], cand['site_id'], cand['item_id'], cand['metric'], cand['as_of'],
                cand['candidate_ea'], 'SERVICE_1', cand['reason'], sv, cand['candidate_version'], sqs,
            )
            ledger.append({**cand, 'db_saved': ok, 'db_message': msg})
        else:
            ledger.append({**cand, 'db_saved': False, 'db_message': 'NO_CANDIDATE'})
    (out / 'proposal_ledger.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in ledger) + '\n', encoding='utf-8')
    master_cnt = wf.master_count()
    assert master_cnt == 0
    return [{'ledger_targets': len(ledger), 'db_proposed': sum(1 for x in ledger if x['db_saved']), 'no_candidate': sum(1 for x in ledger if not x['db_saved']), 'approved_master_count': master_cnt, 'external_write': wf.policy['external_write']}]

# STEP 03 승인과 반려 및 대기의 네 경로 실행하기
def m07_s3(c):
    """승인과 반려 및 대기의 네 경로 실행하기."""
    import sqlite3
    out = folder(c, 7)
    cases = read_json(c['root'] / 'data/approval_cases.json')
    schema_sql = (folder(c, 3) / 'schema.sql').read_text(encoding='utf-8')
    ApprovalWorkflow = c['ApprovalWorkflow']

    results = []
    with tempfile.TemporaryDirectory() as tmp:
        tdb = Path(tmp) / 'approval_test.db'
        conn = sqlite3.connect(tdb)
        conn.executescript(schema_sql)
        conn.close()
        wf = ApprovalWorkflow(tdb, c['root'] / 'config/approval_policy.json', c['root'] / 'config/actor_registry.json')

        for idx, case in enumerate(cases, 1):
            cid = case['case_id']
            wf.propose(cid, 'U001', 'I001', 'on_hand', f'2026-10-01T0{idx}:00:00+00:00', case['candidate_ea'],
                       case['proposer_id'], case['reason'], case['source_versions'], case['candidate_version'], case['quantity_ea'])
            if case['staff_review_present']:
                wf.review_high_impact(cid, case['staff_id'], f'참모 고영향 확인 {cid}')
            if case['requested_action'] in ('approve', 'reject'):
                ok, actual = wf.decide(cid, case['steward_id'], case['requested_action'], case['reason'],
                                       expected_version=case['candidate_version'], current_source_versions=case['source_versions'])
            else:
                ok, actual = True, 'PROPOSED'
            results.append({
                'case_id': cid, 'expected_state': case['expected_state'], 'actual_state': actual,
                'passed': actual == case['expected_state'], 'external_write': case['external_write'],
            })
        assert wf.master_count() == 2
    (out / 'decision_ledger.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in results) + '\n', encoding='utf-8')
    assert c['approval'].master_count() == 0
    c['normal_approval_tests'] = results
    return results

# STEP 04 승인 조건을 어긴 열 가지 요청 차단하기
def m07_s4(c):
    """승인 조건을 어긴 열 가지 요청 차단하기."""
    import sqlite3
    out = folder(c, 7)
    attempts = read_json(c['root'] / 'data/approval_attempts.json')
    schema_sql = (folder(c, 3) / 'schema.sql').read_text(encoding='utf-8')
    ApprovalWorkflow = c['ApprovalWorkflow']
    rows = []

    with tempfile.TemporaryDirectory() as tmp:
        for i, att in enumerate(attempts, 1):
            tdb = Path(tmp) / f'att_{i}.db'
            conn = sqlite3.connect(tdb)
            conn.executescript(schema_sql)
            conn.close()
            wf = ApprovalWorkflow(tdb, c['root'] / 'config/approval_policy.json', c['root'] / 'config/actor_registry.json')
            sv = {'A': 1, 'B': 1, 'C': 1}
            cid = f"C-{att['attempt_id']}"
            high_q = [90, 210, 210] if att['attempt_id'] == 'ATT04' else [60, 60, 60]
            wf.propose(cid, 'U001', 'I001', 'on_hand', '2026-10-01T00:00:00+00:00', 60, 'ANALYST_1', '초기 제안', sv, 1, high_q)

            if att['attempt_id'] == 'ATT07':
                wf.decide(cid, 'STEWARD_1', 'approve', '선행 승인', 1, sv)
            elif att['attempt_id'] == 'ATT08':
                wf.decide(cid, 'STEWARD_1', 'reject', '선행 반려', 1, sv)

            before_cnt = wf.master_count()
            aid = att['attempt_id']
            if aid == 'ATT01':
                ok, msg = wf.decide(cid, 'VIEWER_1', 'approve', '시도', 1, sv)
            elif aid == 'ATT02':
                wf.decide('SELF-1', 'ANALYST_1', 'approve', '본인승인', 1, sv)
                wf.propose('SELF-2', 'U001', 'I001', 'on_hand', '2026-10-01T01:00:00+00:00', 60, 'ANALYST_1', '제안', sv, 1, [60, 60, 60])
                ok, msg = wf.decide('SELF-2', 'ANALYST_1', 'approve', '본인승인', 1, sv)
            elif aid == 'ATT03':
                ok, msg = wf.decide(cid, 'STEWARD_1', 'approve', '', 1, sv)
            elif aid == 'ATT04':
                ok, msg = wf.decide(cid, 'STEWARD_1', 'approve', '참모누락', 1, sv)
            elif aid == 'ATT05':
                ok, msg = wf.decide(cid, 'STEWARD_1', 'approve', '옛버전', 0, sv)
            elif aid == 'ATT06':
                ok, msg = wf.decide(cid, 'STEWARD_1', 'approve', '원관측변경', 1, {'A': 2, 'B': 1, 'C': 1})
            elif aid == 'ATT07':
                ok, msg = wf.decide(cid, 'STEWARD_1', 'approve', '중복승인', 1, sv)
            elif aid == 'ATT08':
                ok, msg = wf.decide(cid, 'STEWARD_1', 'approve', '반려후재승인', 1, sv)
            elif aid == 'ATT09':
                ok, msg = wf.decide(cid, 'VIEWER_1', 'approve', '역할위조', 1, sv, claimed_role='steward')
            elif aid == 'ATT10':
                ok, msg = wf.decide(cid, 'SERVICE_1', 'approve', '자동승인', 1, sv)
            after_cnt = wf.master_count()
            denied_logged = any(r['result'] == 'DENIED' for r in wf.audit_records)
            rows.append({
                'attempt_id': aid, 'scenario': att['scenario'],
                'rejected': not ok, 'master_unchanged': (before_cnt == after_cnt),
                'audit_denied_logged': denied_logged, 'reason': msg,
            })

    assert all(r['rejected'] and r['master_unchanged'] and r['audit_denied_logged'] for r in rows)
    csv_out(out / 'approval_attack_results.csv', rows)
    c['approval_attacks'] = rows
    return rows

# STEP 05 감사 기록의 수정과 삭제 탐지하기
def m07_s5(c):
    """감사 기록의 수정과 삭제 탐지하기."""
    out = folder(c, 7)
    records = list(c['approval'].audit_records)
    anchor = {'count': len(records), 'last_hash': records[-1]['hash'] if records else ('0' * 64)}
    (out / 'audit.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in records) + '\n', encoding='utf-8')
    write_json(out / 'audit_anchor.json', anchor)

    def audit_verify_fn(recs, anc):
        prev = '0' * 64
        for idx, r in enumerate(recs):
            if r.get('prev_hash') != prev:
                return False, f'prev_hash mismatch at seq {idx + 1}'
            body = {k: v for k, v in r.items() if k != 'hash'}
            if sha(body) != r.get('hash'):
                return False, f'hash mismatch at seq {idx + 1}'
            prev = r['hash']
        if len(recs) != anc['count'] or prev != anc['last_hash']:
            return False, 'anchor count or last_hash mismatch'
        return True, 'OK'

    ok_orig, msg_orig = audit_verify_fn(records, anchor)
    tampered = copy.deepcopy(records)
    tampered[len(tampered) // 2]['reason'] = '임의 변조 사유'
    ok_tamp, msg_tamp = audit_verify_fn(tampered, anchor)

    del_mid = copy.deepcopy(records)
    del del_mid[len(del_mid) // 2]
    ok_mid, msg_mid = audit_verify_fn(del_mid, anchor)

    del_last = copy.deepcopy(records[:-1])
    ok_last, msg_last = audit_verify_fn(del_last, anchor)

    tests = [
        {'case': 'valid_original', 'expected_valid': True, 'actual_valid': ok_orig, 'detail': msg_orig, 'passed': ok_orig is True},
        {'case': 'tamper_reason', 'expected_valid': False, 'actual_valid': ok_tamp, 'detail': msg_tamp, 'passed': ok_tamp is False},
        {'case': 'delete_middle', 'expected_valid': False, 'actual_valid': ok_mid, 'detail': msg_mid, 'passed': ok_mid is False},
        {'case': 'delete_last', 'expected_valid': False, 'actual_valid': ok_last, 'detail': msg_last, 'passed': ok_last is False},
    ]
    assert all(t['passed'] for t in tests)
    write_json(out / 'tamper_test.json', {'limit_note': '이력·기준점 동시 재작성 공격까지 단독 방어하지 못하는 교육용 탐지 장치임', 'cases': tests})
    (out / 'review_notes.md').write_text(
        '# 미션7 검토 기록 (review_notes.md)\n'
        '- 65개 비일치 대상 후보 대장(proposal_ledger.jsonl) 및 기본 integration.db 승인 마스터 0건·외부 연동 NOT_CONNECTED 유지 확인.\n'
        '- 격리 시험 DB에서 4개 정상/반려/대기 경로 및 10종 비정상 승인 요청 차단(DENIED 감사 로그 기록) 검증 완료.\n'
        '- SHA256 해시 체인 검증에서 원본 통과 및 사유 변조·중간 삭제·끝 삭제(audit_anchor.json 대조) 3종 모두 탐지 확인.\n',
        encoding='utf-8',
    )
    c['audit_tests'] = tests
    c['audit_verify_fn'] = audit_verify_fn
    return tests

result_html = run_mission(c, 7, [m07_s1, m07_s2, m07_s3, m07_s4, m07_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
